# 第 3 章：Pandas 資料選取、排序與品質檢查

本 Notebook 融合 `lesson03.ipynb` 與 `practice_ch03.py`，並修復原 Notebook 中的中文編碼亂碼。

**適合對象**：已認識 DataFrame，準備學習常用資料整理與品質檢查的初學者。

**學習目標**：
- 檢視多張資料表的規模、欄位與資料型別。
- 使用 `.loc`、`.iloc`、`head()` 與 `tail()` 選取資料。
- 使用單一及多個條件篩選資料。
- 轉換日期、排序資料並建立新欄位。
- 建立欄位健康表，找出缺失率或型別有風險的欄位。

## 學習流程

1. 載入資料並建立資料表字典
2. 檢視資料規模與欄位資訊
3. 使用 `.loc` 與 `.iloc` 選取資料
4. 條件篩選與多條件組合
5. 日期轉換與排序
6. 建立訂單品項營收欄位
7. 建立工作階段欄位健康表
8. 標記高風險欄位並提出處理建議

## 1. 環境設定與資料載入

使用 `common.py` 的共用函式載入課程資料，可避免 Notebook 執行位置不同而找不到 CSV。

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

from common import ensure_packages, load_data

ensure_packages()
data = load_data()

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

customers = data["customers"]
products = data["products"]
orders = data["orders"]
order_items = data["order_items"]
sessions = data["sessions"]
events = data["events"]
ab_assignments = data["ab_assignments"]

tables = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "sessions": sessions,
    "events": events,
    "ab_assignments": ab_assignments,
}

print("資料載入完成。")

資料載入完成。


## 2. 檢視資料表規模

將資料表放入字典後，可以用迴圈一次整理每張表的列數與欄位數。這比逐張執行 `shape` 更容易比較。

In [3]:
table_summary = pd.DataFrame(
    [
        {"資料表": name, "列數": len(df), "欄位數": df.shape[1]}
        for name, df in tables.items()
    ]
)
display(table_summary)

,資料表,列數,欄位數
0,customers,2500,5
1,products,60,3
2,orders,22000,5
3,order_items,39627,5
4,sessions,70000,7
5,events,232067,6
6,ab_assignments,2500,3


### 2.1 查看欄位型別與缺失情況

`DataFrame.info()` 會顯示欄位名稱、非空值數量、資料型別與記憶體用量。原始 Notebook 使用 `print(df.info())`，但 `info()` 本身已經會輸出內容且回傳 `None`，因此不需要再包一層 `print()`。

In [4]:
# 以 orders 為例，避免一次輸出七張表造成畫面過長。
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   order_id      22000 non-null  int64
 1   customer_id   22000 non-null  int64
 2   order_date    22000 non-null  str  
 3   status        22000 non-null  str  
 4   payment_type  22000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 859.5 KB


## 3. 使用 `.loc` 選取資料

`.loc[列, 欄]` 使用「標籤名稱」選取資料。冒號 `:` 表示所有列，欄位清單則決定顯示哪些欄。

In [5]:
# 選取所有列，只顯示 experiment_group 與 customer_id。
selected_by_label = ab_assignments.loc[
    :, ["experiment_group", "customer_id"]
]
display(selected_by_label.head())

,experiment_group,customer_id
0,B,1
1,B,2
2,A,3
3,A,4
4,A,5


## 4. 使用 `.iloc` 選取資料

`.iloc[列, 欄]` 使用從 0 開始的位置編號。`100:105` 會取得第 100 到 104 的位置，不包含結尾 105。欄位 `[2, 0]` 表示先顯示第 3 欄，再顯示第 1 欄。

In [6]:
selected_by_position = ab_assignments.iloc[100:105, [2, 0]]
display(selected_by_position)

,assign_date,customer_id
100,2024-01-11,101
101,2024-01-08,102
102,2024-01-10,103
103,2024-01-07,104
104,2024-01-02,105


### `.loc` 和 `.iloc` 的差別

- `.loc`：使用列標籤或欄位名稱，適合可讀性高的正式分析。
- `.iloc`：使用整數位置，適合依位置快速取樣。

如果欄位順序可能改變，建議使用 `.loc` 或直接使用欄位名稱。

## 5. 使用 `head()` 與 `tail()` 預覽資料

`head()` 顯示前幾列，`tail()` 顯示最後幾列。預設都是 5 列。

In [7]:
order_columns = ["order_id", "status", "customer_id", "order_date"]

print("前 5 筆訂單：")
display(orders[order_columns].head())

print("最後 5 筆訂單：")
display(orders[order_columns].tail())

前 5 筆訂單：


,order_id,status,customer_id,order_date
0,1,completed,2323,2025-05-02
1,2,completed,117,2024-07-14
2,3,cancelled,160,2025-03-29
3,4,completed,1240,2024-07-08
4,5,completed,714,2024-08-12


最後 5 筆訂單：


,order_id,status,customer_id,order_date
21995,21996,completed,2390,2025-08-06
21996,21997,completed,2467,2025-10-04
21997,21998,refunded,1072,2024-07-05
21998,21999,completed,1977,2025-03-31
21999,22000,completed,843,2024-09-06


## 6. 條件篩選

比較運算會產生一個由 `True`／`False` 組成的 Series，再交給 `.loc` 保留條件為 `True` 的列。使用 `.copy()` 可明確建立獨立資料，避免後續修改出現 `SettingWithCopyWarning`。

In [8]:
completed_orders = orders.loc[
    orders["status"] == "completed"
].copy()

print(f"已完成訂單：{len(completed_orders):,} 筆")
display(completed_orders.head())

已完成訂單：20,413 筆


,order_id,customer_id,order_date,status,payment_type
0,1,2323,2025-05-02,completed,wallet
1,2,117,2024-07-14,completed,wallet
3,4,1240,2024-07-08,completed,atm
4,5,714,2024-08-12,completed,atm
5,6,2005,2025-01-14,completed,card


### 6.1 多條件篩選

下面同時要求訂單狀態為 `completed`，而且付款方式為 `atm`。Pandas 使用 `&` 表示逐列的「而且」，每個條件都必須加括號。

In [9]:
completed_atm_orders = orders.loc[
    (orders["status"] == "completed")
    & (orders["payment_type"] == "atm")
].copy()

print(f"已完成且使用 ATM 付款的訂單：{len(completed_atm_orders):,} 筆")
display(completed_atm_orders.head())

已完成且使用 ATM 付款的訂單：5,111 筆


,order_id,customer_id,order_date,status,payment_type
3,4,1240,2024-07-08,completed,atm
4,5,714,2024-08-12,completed,atm
7,8,671,2024-04-22,completed,atm
9,10,211,2024-04-04,completed,atm
10,11,2091,2025-09-01,completed,atm


> 注意：不能使用 Python 的 `and` 連接兩個 Pandas Series。`and` 只判斷單一布林值；逐列條件要使用 `&`。

## 7. 日期轉換與排序

CSV 讀入後，日期欄位通常是 `object` 字串。先用 `pd.to_datetime()` 轉成日期型別，才能正確排序、取得年份或計算時間差。

In [10]:
completed_orders["order_date"] = pd.to_datetime(
    completed_orders["order_date"]
)

# ascending=False 表示由新到舊排列。
recent_completed = completed_orders.sort_values(
    "order_date", ascending=False
)

display(recent_completed.head(10))
print("order_date 型別：", recent_completed["order_date"].dtype)

,order_id,customer_id,order_date,status,payment_type
20745,20746,2270,2025-12-31,completed,card
13850,13851,891,2025-12-31,completed,atm
3796,3797,1420,2025-12-31,completed,card
15491,15492,885,2025-12-31,completed,cod
13097,13098,1027,2025-12-31,completed,atm
9513,9514,2421,2025-12-31,completed,atm
6084,6085,1309,2025-12-31,completed,wallet
2874,2875,1768,2025-12-31,completed,card
5911,5912,828,2025-12-31,completed,wallet
15891,15892,1786,2025-12-31,completed,cod


order_date 型別： datetime64[us]


## 8. 建立訂單品項營收欄位

Pandas 欄位也支援向量化運算。每筆品項營收為：

`quantity × unit_price × (1 - discount_rate)`

In [11]:
order_items_with_revenue = order_items.copy()
order_items_with_revenue["line_revenue"] = (
    order_items_with_revenue["quantity"]
    * order_items_with_revenue["unit_price"]
    * (1 - order_items_with_revenue["discount_rate"])
)

display(order_items_with_revenue.head())

,order_id,product_id,quantity,unit_price,discount_rate,line_revenue
0,1,28,1,567,0.05,538.65
1,2,2,1,3850,0.05,"3,657.50"
2,2,18,1,778,0.10,700.20
3,3,19,1,2193,0.00,"2,193.00"
4,3,36,1,1580,0.15,"1,343.00"


## 9. 建立欄位健康表

接著融合 `practice_ch03.py`。欄位健康表為 `sessions` 的每個欄位整理三項資訊：

- `dtype`：目前資料型別。
- `missing_rate`：缺失值比例。
- `nunique`：排除缺失值後的唯一值數量。

排序時先看缺失率，再看唯一值數量，讓較值得注意的欄位排在前面。

In [12]:
health = pd.DataFrame({
    "column": sessions.columns,
    "dtype": [str(sessions[column].dtype) for column in sessions.columns],
    "missing_rate": [
        round(sessions[column].isna().mean(), 4)
        for column in sessions.columns
    ],
    "nunique": [
        sessions[column].nunique(dropna=True)
        for column in sessions.columns
    ],
}).sort_values(
    ["missing_rate", "nunique"],
    ascending=[False, False],
).reset_index(drop=True)

display(health)

,column,dtype,missing_rate,nunique
0,session_id,int64,0.00,70000
1,session_start,str,0.00,67778
2,customer_id,int64,0.00,2500
3,traffic_source,str,0.00,5
4,campaign,str,0.00,5
5,device,str,0.00,3
6,experiment_group,str,0.00,2


### 健康表程式的意思

- `sessions.columns`：取得所有欄位名稱。
- `isna().mean()`：`True` 會視為 1，因此平均值就是缺失比例。
- `nunique(dropna=True)`：計算非缺失的不同值數量。
- `sort_values()`：依指定欄位排序。
- `reset_index(drop=True)`：排序後重新產生連續索引。

## 10. 標記高風險欄位

本練習使用兩條示範規則：

1. 缺失率超過 20%。
2. 欄位名稱包含 `date`、`time`、`start` 或 `end`，但型別仍是文字型別（`object`、`str` 或 `string`）。

兩條規則只要符合其中一條就列為風險，因此使用 `|` 表示逐列的「或者」。

In [13]:
high_missing = health["missing_rate"] > 0.2
looks_like_datetime = health["column"].str.contains(
    "date|time|start|end", case=False, regex=True
)
# 不同 Pandas 版本可能把文字欄位顯示為 object、str 或 string。
stored_as_text = health["dtype"].str.lower().isin(["object", "str", "string"])

risks = health.loc[
    high_missing | (looks_like_datetime & stored_as_text),
    ["column", "dtype", "missing_rate", "nunique"],
].copy()

if risks.empty:
    print("依目前規則，沒有高風險欄位。")
else:
    display(risks)

,column,dtype,missing_rate,nunique
1,session_start,str,0.00,67778


## 11. 產生欄位處理建議

以下規則只是通用起點，實際處理前仍需確認業務意義：

- 缺失率超過 40%：評估刪除欄位，或建立明確的「未知」類別。
- 缺失率大於 0：考慮填補，或新增缺失指標欄位。
- 無缺失：目前可保留。
- 名稱像日期／時間且型別為 `object`：再加上日期轉換建議。

In [14]:
def suggest_strategy(row: pd.Series) -> str:
    """依欄位健康資訊產生初步處理建議。"""
    column = row["column"]
    missing_rate = row["missing_rate"]
    dtype = row["dtype"]

    if missing_rate > 0.4:
        action = "經業務確認後，評估刪除或建立未知類別"
    elif missing_rate > 0:
        action = "考慮填補，或新增缺失指標欄位"
    else:
        action = "目前可保留"

    text_dtypes = {"object", "str", "string"}
    datetime_keywords = ("date", "time", "start", "end")
    if any(keyword in column.lower() for keyword in datetime_keywords) and dtype.lower() in text_dtypes:
        action += "；轉換為 datetime"

    return action


health_with_strategy = health.copy()
health_with_strategy["suggested_strategy"] = health_with_strategy.apply(
    suggest_strategy, axis=1
)
display(health_with_strategy)

,column,dtype,missing_rate,nunique,suggested_strategy
0,session_id,int64,0.00,70000,目前可保留
1,session_start,str,0.00,67778,目前可保留；轉換為 datetime
2,customer_id,int64,0.00,2500,目前可保留
3,traffic_source,str,0.00,5,目前可保留
4,campaign,str,0.00,5,目前可保留
5,device,str,0.00,3,目前可保留
6,experiment_group,str,0.00,2,目前可保留


## 12. 實際轉換工作階段日期

風險表指出名稱像時間但仍是字串的欄位後，可在副本中使用 `pd.to_datetime()` 轉換，再確認型別。

In [15]:
sessions_clean = sessions.copy()
sessions_clean["session_start"] = pd.to_datetime(
    sessions_clean["session_start"], errors="coerce"
)

print("轉換前型別：", sessions["session_start"].dtype)
print("轉換後型別：", sessions_clean["session_start"].dtype)
print("轉換後無法解析的筆數：", sessions_clean["session_start"].isna().sum())

轉換前型別： str
轉換後型別： datetime64[us]
轉換後無法解析的筆數： 0


## 13. 練習題

請替 `orders` 建立欄位健康表，並找出：

1. 哪些欄位含有缺失值？
2. 哪些日期或時間欄位仍是 `object`？
3. 轉換 `order_date` 後，最新一筆已完成訂單是哪一天？

In [16]:
# TODO：可先遮住以下參考答案，再自行完成。
orders_health = pd.DataFrame({
    "column": orders.columns,
    "dtype": [str(orders[column].dtype) for column in orders.columns],
    "missing_rate": [orders[column].isna().mean() for column in orders.columns],
    "nunique": [orders[column].nunique(dropna=True) for column in orders.columns],
})

orders_for_exercise = orders.copy()
orders_for_exercise["order_date"] = pd.to_datetime(
    orders_for_exercise["order_date"], errors="coerce"
)
latest_completed_date = orders_for_exercise.loc[
    orders_for_exercise["status"] == "completed", "order_date"
].max()

display(orders_health)
print("最新已完成訂單日期：", latest_completed_date.date())

,column,dtype,missing_rate,nunique
0,order_id,int64,0.00,22000
1,customer_id,int64,0.00,2499
2,order_date,str,0.00,731
3,status,str,0.00,3
4,payment_type,str,0.00,4


最新已完成訂單日期： 2025-12-31


## 常見錯誤與延伸

**常見錯誤**：
- 把 `.loc` 與 `.iloc` 混用：前者使用標籤，後者使用位置。
- 使用 `print(df.info())`，導致資訊後方多印一個 `None`。
- 多條件篩選使用 `and`／`or`；Pandas Series 應使用 `&`／`|`。
- 在篩選結果上直接賦值而沒有 `.copy()`，可能出現 `SettingWithCopyWarning`。
- 日期仍是字串就直接排序；字串格式不一致時可能得到錯誤順序。

**延伸練習**：
- 對七張資料表批次建立健康表，再合併成一張完整品質報告。
- 為唯一值過少或過多的欄位新增風險規則。
- 比較不同付款方式的訂單數與營收。

## 重點整理

- `.loc` 使用標籤，`.iloc` 使用整數位置。
- 布林條件可搭配 `.loc` 精確篩選資料。
- 日期轉換後才能可靠地排序與進行時間分析。
- 欄位健康表能快速彙整型別、缺失率與唯一值數量。
- 自動建議只能輔助判斷，正式清理仍需結合業務規則。